# BF4 — Clustering & Collaborative Filtering

1. **Clustering**: Ward-linkage on 28-dim semantic features. Also uses BIRD's `semantic_info_entire_trajectories.csv` for comparison.
2. **Collaborative filtering**: User-item matrix from `explicit_feedback_visitors.csv`. Simple user-based CF with cosine similarity.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.abspath('helpers'))
from fuseki_config import PROJECT_ROOT
from semantic_features import SemanticFeatureExtractor, normalize_features
import json

## Part 1: Trajectory Clustering

In [ ]:
# Load pre-extracted features (from BF3) or extract fresh
features_path = Path('output/semantic_features/trajectory_features.json')
if features_path.exists():
    with open(features_path) as f:
        features_json = json.load(f)
    features = {uri: np.array(vec, dtype=np.float32) for uri, vec in features_json.items()}
    print(f'Loaded {len(features)} trajectory features from cache')
else:
    extractor = SemanticFeatureExtractor()
    features = extractor.batch_extract_features()

uris = list(features.keys())
X = np.array([features[uri] for uri in uris])
print(f'Feature matrix: {X.shape}')

In [ ]:
# Z-normalize per feature group
extractor_tmp = SemanticFeatureExtractor()
groups = extractor_tmp.get_feature_groups()
norm_features = normalize_features(features, groups)
X_norm = np.array([norm_features[uri] for uri in uris])

# Ward linkage clustering
Z = linkage(X_norm, method='ward')

# Dendrogram
fig, ax = plt.subplots(figsize=(14, 6))
labels = [uri.split('_')[-1] if '_' in uri else uri[-8:] for uri in uris]
dendrogram(Z, labels=labels, leaf_rotation=90, leaf_font_size=8, ax=ax)
ax.set_title('Trajectory Clustering (Ward Linkage, 28-dim Semantic Features)')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()

In [ ]:
# Cut into clusters
n_clusters = 4
cluster_labels = fcluster(Z, n_clusters, criterion='maxclust')
print(f'Cluster distribution (k={n_clusters}):')
for c in range(1, n_clusters + 1):
    members = [uris[i] for i in range(len(uris)) if cluster_labels[i] == c]
    print(f'  Cluster {c}: {len(members)} trajectories')

### Comparison: BIRD Semantic Info Clustering

In [ ]:
# Load BIRD's own semantic info for comparison clustering
sem_path = PROJECT_ROOT / 'data' / 'bird' / 'bird-dataset-main' / 'data' / 'semantic_info_entire_trajectories.csv'
sem_info = pd.read_csv(sem_path)
print(f'BIRD semantic info: {len(sem_info)} trajectories')

# Cluster on BIRD features: duration, speed, nb_items, nb_stops, length, curvature
bird_features = sem_info[['duration', 'speed', 'nb_items', 'nb_stops', 'length', 'curvature']].values
scaler = StandardScaler()
bird_norm = scaler.fit_transform(bird_features)

Z_bird = linkage(bird_norm, method='ward')
bird_clusters = fcluster(Z_bird, n_clusters, criterion='maxclust')

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z_bird, labels=sem_info['trajectory_id'].astype(str).values,
           leaf_rotation=90, leaf_font_size=8, ax=ax)
ax.set_title('BIRD Semantic Info Clustering (speed, duration, items, stops)')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()

print('\nBIRD cluster distribution:')
for c in range(1, n_clusters + 1):
    print(f'  Cluster {c}: {(bird_clusters == c).sum()} trajectories')

## Part 2: Collaborative Filtering

In [ ]:
# Load explicit feedback
feedback_path = PROJECT_ROOT / 'data' / 'bird' / 'bird-dataset-main' / 'data' / 'explicit_feedback_visitors.csv'
feedback = pd.read_csv(feedback_path)
print(f'Feedback records: {len(feedback)}')
print(f'Unique visitors: {feedback["visitor_id"].nunique()}')
print(f'Unique items: {feedback["item_id"].nunique()}')
feedback.head()

In [ ]:
# Build user-item binary matrix
# 1 = visitor viewed/liked this artwork, 0 = no interaction
user_item = feedback.pivot_table(
    index='visitor_id', columns='item_id',
    aggfunc='size', fill_value=0
)
user_item = (user_item > 0).astype(int)  # Binarize
print(f'User-item matrix: {user_item.shape}')
print(f'Sparsity: {1 - user_item.values.sum() / (user_item.shape[0] * user_item.shape[1]):.3f}')

In [ ]:
# User-based collaborative filtering with cosine similarity
user_sim = cosine_similarity(user_item.values)
user_sim_df = pd.DataFrame(user_sim, index=user_item.index, columns=user_item.index)

print('User similarity matrix shape:', user_sim_df.shape)
print(f'Mean user similarity: {user_sim[np.triu_indices_from(user_sim, k=1)].mean():.4f}')

In [ ]:
def cf_recommend(visitor_id, user_item_matrix, user_sim_df, top_k=10, k_neighbors=5):
    """Recommend artworks for a visitor using user-based CF."""
    if visitor_id not in user_sim_df.index:
        return []

    # Find k most similar users
    sim_scores = user_sim_df.loc[visitor_id].drop(visitor_id)
    top_neighbors = sim_scores.nlargest(k_neighbors)

    # Items the visitor hasn't seen
    visited = set(user_item_matrix.columns[user_item_matrix.loc[visitor_id] > 0])

    # Score unvisited items by neighbor similarity
    item_scores = {}
    for neighbor, sim in top_neighbors.items():
        neighbor_items = set(user_item_matrix.columns[user_item_matrix.loc[neighbor] > 0])
        for item in neighbor_items - visited:
            item_scores[item] = item_scores.get(item, 0) + sim

    ranked = sorted(item_scores.items(), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# Test CF for a sample visitor
sample_visitor = user_item.index[0]
recs = cf_recommend(sample_visitor, user_item, user_sim_df)
print(f'CF recommendations for visitor {sample_visitor}:')
for item, score in recs:
    print(f'  {item}: {score:.4f}')

In [ ]:
# Leave-one-out evaluation on trajectory visitors
# Only evaluate visitors who appear in both feedback and trajectories
traj_visitors = set(sem_info['trajectory_id'].values)
feedback_visitors = set(feedback['visitor_id'].values)
shared_visitors = traj_visitors & feedback_visitors
print(f'Visitors in both trajectories and feedback: {len(shared_visitors)}')

if len(shared_visitors) > 5:
    hits = 0
    total = 0

    for visitor in shared_visitors:
        if visitor not in user_item.index:
            continue

        visitor_items = list(user_item.columns[user_item.loc[visitor] > 0])
        if len(visitor_items) < 2:
            continue

        # Hold out last item
        held_out = visitor_items[-1]

        # Temporarily remove from matrix
        user_item_loo = user_item.copy()
        user_item_loo.loc[visitor, held_out] = 0

        # Recompute similarity for this visitor
        user_sim_loo = cosine_similarity(user_item_loo.values)
        user_sim_loo_df = pd.DataFrame(user_sim_loo, index=user_item_loo.index, columns=user_item_loo.index)

        recs = cf_recommend(visitor, user_item_loo, user_sim_loo_df, top_k=20)
        rec_items = [item for item, _ in recs]

        if held_out in rec_items:
            hits += 1
        total += 1

    hit_rate = hits / total if total > 0 else 0
    print('\nLeave-one-out evaluation:')
    print(f'  Hit@20: {hits}/{total} = {hit_rate:.3f}')
else:
    print('Not enough shared visitors for evaluation')